In [ ]:
# Path handling (standard library)
from pathlib import Path

# Core imports
import ionerdss as ion
from ionerdss import build_system_from_pdb

# For visualizations
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pdb_id = "4yd9"

# Build the system using simplified API
# This should take ~10 seconds for 6bno
system = build_system_from_pdb(
    source=pdb_id,
    workspace_path=f"{pdb_id}_dir",
    # Interface detection
    #interface_detect_distance_cutoff=1.0,  # Standard cutoff for protein interfaces
    interface_detect_distance_cutoff=0.6,
    interface_detect_n_residue_cutoff=3,
    chain_grouping_seq_threshold=0.5,
    nerdss_water_box=[500.0, 500.0, 500.0],
    
    # ODE Pipeline Configuration
    ode_enabled=True,            # Now using System-compatible generator!
    ode_time_span=None,   # Auto-calculated based on NERDSS simulation time
    ode_solver_method="BDF",     # Solver for stiff systems
    ode_plot=True,               # Generate plots
    ode_save_csv=True,            # Save data to CSV

    # Transition matrix parameters
    count_transition=True,        # Enable transition matrix tracking
    transition_matrix_size=75,   # Size of matrix (max cluster size expected)
    transition_write=1000,         # Write every 1000 iterations
)

In [ ]:
# List all generated files
workspace_path = Path(f"{pdb_id}_dir")

print("Generated Files:")
print("\n ODE Results:")
ode_dir = workspace_path / "ode_results"
if ode_dir.exists():
    for file in sorted(ode_dir.glob("*")):
        size = file.stat().st_size / 1024  # KB
        print(f"  ode_results/{file.name:<31} ({size:>6.1f} KB)")

print("\n NERDSS Input Files:")
nerdss_dir = workspace_path / "nerdss_files"
if nerdss_dir.exists():
    for file in sorted(nerdss_dir.glob("*.mol")) + sorted(nerdss_dir.glob("*.inp")):
        size = file.stat().st_size / 1024  # KB
        print(f"  nerdss_files/{file.name:<30} ({size:>6.1f} KB)")

print("\n System Data:")
outputs_dir = workspace_path / "outputs" / "systems"
if outputs_dir.exists():
    for file in sorted(outputs_dir.glob("*.json")):
        size = file.stat().st_size / 1024  # KB
        print(f"  outputs/systems/{file.name:<27} ({size:>6.1f} KB)")

print("\n System Builder Log:")
outputs_dir = workspace_path / "logs"
if outputs_dir.exists():
    for file in sorted(outputs_dir.glob("*.log")):
        size = file.stat().st_size / 1024  # KB
        print(f"  logs/{file.name:<38} ({size:>6.1f} KB)") 

In [ ]:
# run NERDSS with subprocess
import subprocess

# Check if NERDSS is available
# nerdss_cmd should be replaced with the actual path to the NERDSS executable
nerdss_cmd = "PATH_TO_NERDSS_REPO/bin/nerdss"
nerdss_path = Path(nerdss_cmd).expanduser() # replaces tilde with appropriate user home path

if nerdss_path.exists():
    
    # Run NERDSS
    result = subprocess.run(
        f"{nerdss_cmd} -f parms.inp",
        shell=True,
        cwd=f"{pdb_id}_dir/nerdss_files",
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✓ NERDSS simulation completed!")
        print(f"\nCheck {pdb_id}_dir/nerdss_files/ for output files")
    else:
        print("⚠ NERDSS simulation failed")
        print(result.stderr[:500])
else:
    print("⚠ NERDSS not found at:", nerdss_cmd)

In [ ]:
# Initialize Analyzer with NERDSS output directory
analysis = ion.Analyzer(f"{pdb_id}_dir")

# Display discovered simulations
print(f"Found {len(analysis.simulations)} simulation(s)")
for i, sim in enumerate(analysis.simulations):
    print(f"  [{i}] Simulation ID: {sim.id}")

#################
# Plot the NERDSS trajectory alone
plt.figure()

sim = analysis.get_simulation(0)
complex_compositions = [{"A":n} for n in range(1,11)] # A1 through A10

# get the time series data for the above complexes
time, counts = sim.get_time_series(complex_compositions)

# plot all the returned data
for i in range(10):
    plt.plot(time,counts[i],label=str(complex_compositions[i]))

plt.legend()
plt.show()

################
# plot ODE and NERDSS side to side 
# Specify the path to your csv file
csv_path = f'./{pdb_id}_dir/ode_results/ode_solution_simple.csv'

# Display the image
df = pd.read_csv(csv_path)

# Create a new figure
plt.figure()

# Only plot first n species
plot_first_n_species = 4

# plot nerdss simulation
for i in range(plot_first_n_species):
    plt.plot(time, counts[i], label=str(f"A{i+1}(sim)"))

# ode results
for col in df.columns:
    # plot first 
    if col != "time":
        # df contains concentration in uM, convert to counts
        # 1 uM ~= 75 in 500 nm x 500 nm x 500 nm box
        # Change this if you used a different initial count
        initial_count_number = 75
        plt.plot(df["time"], df[col] * initial_count_number,
            label = (f"{col}(ode)"), ls = "--")

plt.legend()
plt.show()